# T-REX: the source through the chopper train

The ESS source T-REX sees, as neutrons per second into its guide's entrance over
(emission time, wavelength): McStas's `ESS_butterfly` moderator for beamline W7, computed
by `ess_butterfly.py`. chopcal transmits the source rectangle through T-REX's six discs
exactly, as convex polygons, and weights it by how likely each neutron is to get through
while a disc's edge is still sweeping the beam.

The discs are phased as `trex_primary.instr` phases them. Each delay is the time a
neutron of the central wavelength L0, emitted 1.7 ms into the pulse, takes to reach the
disc. The bandwidth pair turns at 14 Hz, the pulse-shaping (P) pair against each other
at ¾ × RRM × 14 Hz, and the monochromating (M) pair at RRM × 14 Hz. HF opens the P and M
discs' wider slits, HR their narrower ones. The P pair and M1 hang above the beam, the
rest below it. M1 and M2 turn the same way, but on opposite sides of the beam, so their
edges cross it in opposite directions: to the beam they are a counter-rotating pair.
In the instr both of M1's openings line up with M2's in either mode, as though one disc
were mounted the other way round from the design. *M2 turned over* mirrors M2's
openings, still phased to put the selected slit on the beam, so only the selected pair
lines up. Most of the time the P pair hides the difference; it shows with the P pair
parked, or at high RRM. Against McStas running the instr, the fraction of the beam the M pair passes agrees
within counting error (HF 4.1 %, HR 2.9 %).

Repetition-rate multiplication puts several bands through each pulse, RRM × 14 Hz
apart in time at the M pair and so about 3956 / (14 × RRM × 162) Å apart in wavelength.
All of those emitted during the proton pulse are intended, outlined in red. Any region
emitted only after the pulse is spurious, outlined and plotted in purple; use *log
intensities* to see it.

*guide transmission*, on by default, weights what reaches the sample by the fraction of
what enters the guide that reaches a 10 × 30 mm sample window. It comes from McStas
scans of `trex_primary.instr` with the choppers removed, 10⁷ rays per point from 0.5 to
12 Å, one with the bispectral bender out and one with it in; *bender in* picks the
second. The bender moves no disc, only the guide's transmission.

The time-distance diagram's second panel shows the M pair and the sample around the L0
band, with one band either side.

RRM is 4, 8, ... 24: a multiple of 4, so that every disc, the P pair at ¾ × RRM × 14 Hz
among them, turns at a multiple of 14 Hz. The controls set L0, RRM and HF or HR, the source rectangle, the source and arrival grids -- fine by default, since
each band is a narrow strip arriving over a few hundred µs, and σλ is taken per arrival bin -- the proton pulse
and beam power, a factor on the moderator's decay times, and a pulse delay moving every
disc together. Each disc has a delay, which follows the phasing whenever L0, RRM or the
mode changes and otherwise stays where it is put, and a *parked* toggle, which stops it
open.

Needs `chopcal>=0.8`, `ipympl` and `ipywidgets`.


In [ ]:
%matplotlib widget

from phasespace import figure
from trex import TREX

In [ ]:
figure(TREX, wavelength=2.4, rrm=4, mode='HF')

## The guide's transmission

The fraction of what enters T-REX's guide that reaches a 10 × 30 mm sample window,
against wavelength, with the bispectral bender out and in: the tables the *guide
transmission* toggle weights the source by. From McStas scans of `trex_primary.instr`
with the choppers removed, 10⁷ rays per point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from trex import TREX

fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
for bender, path in TREX.guide_tables.items():
    wavelength, fraction, error = np.loadtxt(path, delimiter=',', skiprows=1).T
    ax.errorbar(wavelength, 100 * fraction, 100 * error, fmt='.-', markersize=3,
                label=f"bender {'in' if bender else 'out'}")
ax.set_xlabel('wavelength [Å]')
ax.set_ylabel('guide transmission [%]')
ax.set_ylim(bottom=0)
ax.legend()
plt.show()

## Without widgets

`transmission` computes one setting and `draw` puts it on a figure, for a script or a
saved picture.

In [ ]:
from phasespace import draw, make_axes, transmission
from trex import TREX

result = transmission(TREX, dict(wavelength=2.4, rrm=4, mode='HF'))
fig, axes = make_axes()
draw(result, axes, log=True)
print(f'{result.transmitted:.3g} n/s, {result.transmitted / result.emitted:.3%} of the '
      'source rectangle, reach the sample')